### Compare runs, choose a model and deploy it to a REST API

In [7]:
import keras
import mlflow
from mlflow.tracking import MlflowClient

# Patch module attribute for MLflow compatibility
mlflow.MlflowClient = MlflowClient

import numpy as np
import pandas as pd
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from mlflow.models import infer_signature
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Set tracking URI and experiment
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("/wine-quality")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1785708154877, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1785708154877, lifecycle_stage='active', name='/wine-quality', tags={}, trace_location=None, workspace='default'>

In [8]:
# Load the dataset
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-red.csv",
    sep=";",
)

In [9]:
# Split the data into training, validation, and test sets
train, test = train_test_split(data, test_size=0.25, random_state=42)

# Train
train_x = train.drop(["quality"], axis=1).values
train_y = train["quality"].values.ravel()

# Test
test_x = test.drop(["quality"], axis=1).values
test_y = test["quality"].values.ravel()

# Validation
train_x, val_x, train_y, val_y = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

signature = infer_signature(train_x, train_y)

In [10]:
def train_model(params, epochs, train_x, train_y, val_x, val_y, test_x, test_y):

    # Define model architecture
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    model = keras.Sequential(
        [
            keras.Input(shape=(train_x.shape[1],)),
            keras.layers.Normalization(mean=mean, variance=var),
            keras.layers.Dense(64, activation="relu"),
            keras.layers.Dense(1),
        ]
    )

    # Compile the model
    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"], momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()],
    )

    # Train the model inside a child run
    with mlflow.start_run(nested=True):
        model.fit(
            train_x,
            train_y,
            validation_data=(val_x, val_y),
            epochs=epochs,
            batch_size=64,
            verbose=0,
        )

        # Evaluate the model
        eval_result = model.evaluate(val_x, val_y, batch_size=64, verbose=0)
        eval_rmse = eval_result[1]

        # Log parameters, metrics, and model artifact
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", eval_rmse)
        mlflow.keras.log_model(model, "model", signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [11]:
def objective(params):
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        val_x=val_x,
        val_y=val_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result


# Define hyperparameter search space
space = {
    "lr": hp.loguniform("lr", -11.5129, -2.3025),
    "momentum": hp.uniform("momentum", 0.0, 1.0),
}

In [ ]:
# Run parent optimization pipeline
with mlflow.start_run(run_name="hyperopt_parent"):
    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials,
    )   

    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    mlflow.log_params(best) 
    mlflow.log_metric("eval_rmse", best_run["loss"])

    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")

  0%|          | 0/4 [00:00<?, ?trial/s, best loss=?]

2026/08/03 01:12:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run chill-moth-96 at: http://127.0.0.1:5000/#/experiments/3/runs/55e4ab60acd54f8485b62662e3bcd359

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3

 25%|██▌       | 1/4 [00:12<00:37, 12.51s/trial, best loss: 0.9204815626144409]

2026/08/03 01:12:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run exultant-ox-580 at: http://127.0.0.1:5000/#/experiments/3/runs/4ffd6e60580549ad851da698aa11b971

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

 50%|█████     | 2/4 [00:25<00:25, 12.58s/trial, best loss: 0.9204815626144409]

2026/08/03 01:12:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run agreeable-goat-571 at: http://127.0.0.1:5000/#/experiments/3/runs/4261fa3822c647c881099e19c4ec2f56

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

 75%|███████▌  | 3/4 [00:39<00:13, 13.19s/trial, best loss: 0.9204815626144409]

2026/08/03 01:12:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run adventurous-fox-794 at: http://127.0.0.1:5000/#/experiments/3/runs/738f9c42c0014a6cb77854b728fb47be

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3                   

100%|██████████| 4/4 [00:50<00:00, 12.74s/trial, best loss: 0.7911251187324524]
Best parameters: {'lr': np.float64(0.02173623296025937), 'momentum': np.float64(0.7979677480697503)}
Best eval rmse: 0.7911251187324524
🏃 View run hyperopt_parent at: http://127.0.0.1:5000/#/experiments/3/runs/493b5153eb1647a5a62df091bd107a39
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
